# 04 · Pipeline schedules: GPipe vs 1F1B

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/distributed/04_pipeline_schedules.ipynb) &nbsp; 📖 Concept deck: [Pipeline parallelism](https://nilesh-salpe.github.io/visual-ai/distributed/05-pipeline-parallelism/) &nbsp; ⏱️ Suggested time: **45 min**

**You will build:** a small pipeline simulator that turns per-stage operation orders into a timeline, then GPipe and 1F1B schedules, their bubble and their peak activation memory.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def simulate(orders, p, tf=1.0, tb=2.0):
    """orders[s] = list of ('F' or 'B', microbatch) for stage s, executed in that order.
    F(s,i) needs F(s-1,i); B(s,i) needs B(s+1,i) and F(s,i). Returns (makespan, finish times dict, peak in-flight per stage)."""
    done, ptr, clock = {}, [0] * p, [0.0] * p
    inflight, peak = [0] * p, [0] * p
    progress = True
    while progress:
        progress = False
        for s in range(p):
            if ptr[s] >= len(orders[s]):
                continue
            op, i = orders[s][ptr[s]]
            deps = [('F', s - 1, i)] if op == 'F' and s > 0 else []
            if op == 'B':
                deps = [('F', s, i)] + ([('B', s + 1, i)] if s < p - 1 else [])
            if all(dp in done for dp in deps):
                start = max([clock[s]] + [done[dp] for dp in deps])
                done[(op, s, i)] = clock[s] = start + (tf if op == 'F' else tb)
                inflight[s] += 1 if op == 'F' else -1
                peak[s] = max(peak[s], inflight[s])
                ptr[s] += 1; progress = True
    assert all(ptr[s] == len(orders[s]) for s in range(p)), "deadlock: an order violates dependencies"
    return max(clock), done, peak

## Task 1: GPipe: all forwards, then all backwards

GPipe runs every micro-batch forward through the pipeline, then every backward. For each stage s the order is F0, F1, …, F(m−1), B0, B1, …, B(m−1).
Return `orders` (a list of p lists of `('F', i)` / `('B', i)`).

In [ ]:
def gpipe_orders(p, m):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_gpipe_orders(f):
    p, m = 4, 8
    mk, _, peak = simulate(f(p, m), p)
    assert np.isclose(mk, (m + p - 1) * 3.0), f"GPipe makespan should be (m+p-1)(tf+tb) = {(m + p - 1) * 3.0}, got {mk}"
    assert peak == [m] * p, f"GPipe keeps all m micro-batches' activations: peak {peak}"

check("gpipe_orders", _test_gpipe_orders, gpipe_orders)

## Task 2: 1F1B: start backwards early

PipeDream's 1F1B schedule: stage s does `p − s − 1` warm-up forwards, then alternates one forward and one backward, then finishes the remaining backwards.
Same makespan as GPipe for the same p and m, but each stage holds at most `p − s` micro-batches of activations instead of m.

In [ ]:
def one_f_one_b_orders(p, m):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_one_f_one_b_orders(f):
    p, m = 4, 8
    orders = f(p, m)
    assert all(sorted(o) == sorted([('F', i) for i in range(m)] + [('B', i) for i in range(m)]) for o in orders), "each stage does every F and B once"
    mk, _, peak = simulate(orders, p)
    assert np.isclose(mk, (m + p - 1) * 3.0), f"1F1B makespan should equal GPipe's, got {mk}"
    assert peak == [p - s for s in range(p)], f"peak in-flight should be p - s per stage, got {peak}"

check("one_f_one_b_orders", _test_one_f_one_b_orders, one_f_one_b_orders)

## Task 3: The bubble

Ideal time (no pipeline) for one stage's work is m·(tf + tb). The **bubble fraction** is the extra time relative to that ideal:
$$\text{bubble} = \frac{\text{makespan} - m(t_f+t_b)}{m(t_f+t_b)}$$
Megatron (Narayanan et al., 2021) derives it as $(p-1)/m$, and with an interleaved schedule of v chunks per GPU as $(p-1)/(v\cdot m)$ (also the formula Llama 3 quotes). Compute it from a simulated makespan.

In [ ]:
def bubble_fraction(makespan, m, tf=1.0, tb=2.0):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_bubble_fraction(f):
    for p, m in [(4, 8), (8, 32), (16, 64)]:
        mk, _, _ = simulate(gpipe_orders_solution(p, m) if 'gpipe_orders_solution' in globals() else [[('F', i) for i in range(m)] + [('B', i) for i in range(m)] for _ in range(p)], p)
        assert np.isclose(f(mk, m), (p - 1) / m), f"p={p}, m={m}: expected (p-1)/m = {(p - 1) / m:.3f}"

check("bubble_fraction", _test_bubble_fraction, bubble_fraction)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: GPipe: all forwards, then all backwards
def gpipe_orders_solution(p, m):
    return [[('F', i) for i in range(m)] + [('B', i) for i in range(m)] for _ in range(p)]

check("gpipe_orders (reference)", _test_gpipe_orders, gpipe_orders_solution)

In [ ]:
# @title Solution for Task 2: 1F1B: start backwards early
def one_f_one_b_orders_solution(p, m):
    orders = []
    for s in range(p):
        warm = min(p - s - 1, m)
        o = [('F', i) for i in range(warm)]
        f_next, b_next = warm, 0
        while f_next < m:
            o += [('F', f_next), ('B', b_next)]; f_next += 1; b_next += 1
        o += [('B', i) for i in range(b_next, m)]
        orders.append(o)
    return orders

check("one_f_one_b_orders (reference)", _test_one_f_one_b_orders, one_f_one_b_orders_solution)

In [ ]:
# @title Solution for Task 3: The bubble
def bubble_fraction_solution(makespan, m, tf=1.0, tb=2.0):
    ideal = m * (tf + tb)
    return (makespan - ideal) / ideal

check("bubble_fraction (reference)", _test_bubble_fraction, bubble_fraction_solution)

---
## Experiments
**Bubble vs micro-batches, and the memory trade-off.** More micro-batches shrink the bubble ((p−1)/m) but GPipe must hold all of them; 1F1B caps memory at p in-flight micro-batches on the first stage. Interleaving (v model chunks per GPU) divides the bubble by v at the cost of v× more point-to-point messages.

In [ ]:
p = 8
for m in [8, 16, 32, 64]:
    mk, _, peak_g = simulate(gpipe_orders_solution(p, m), p)
    _, _, peak_1 = simulate(one_f_one_b_orders_solution(p, m), p)
    b = bubble_fraction_solution(mk, m)
    print(f"p={p}, m={m:3d}: bubble {b:5.1%}  (interleaved v=4: {(p - 1) / (4 * m):5.1%})   peak activations stage 0: GPipe {peak_g[0]:3d}, 1F1B {peak_1[0]}")

---
## Takeaways
* A pipeline of p stages idles while it fills and drains: bubble = (p−1)/m of the ideal time, so you need many micro-batches (GPipe: negligible when m ≥ 4p).
* 1F1B has the same bubble as GPipe but bounds activation memory by p micro-batches instead of m.
* Interleaved schedules divide the bubble by v; zero-bubble schedules split the backward pass to fill the remaining gaps.